# Campaign 28 manual split audit

This notebook renders the exact Campaign 28 supervision for its 17 fragment patches.

- **red**: ordinary training positive
- **magenta**: explicit positive (`185` in `train_masks`)
- **yellow**: training ring negative
- **blue**: validation positive
- **green**: validation ring negative
- **cyan**: explicit negative (`119` in `train_masks`)
- **gray**: researcher `inklabels/2_4um` reference

Campaign 28 uses top-level radius-12 dilated labels for supervision with a closed ring (`close=2`, `gap=2`, `shell=4`) and `pos_only=True`. The gray background comes from `inklabels/2_4um`; fragments without labels or train masks remain intentionally unavailable until those repository inputs are added.

In [ ]:
from pathlib import Path
import gc
import importlib

import cv2
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

import utils.config as config_module
import utils.dataloader as dataloader_module

ROOT = Path.cwd()
if ROOT.name == "old":
    ROOT = ROOT.parent

config_module = importlib.reload(config_module)
campaign28_module = importlib.import_module("campaign_archs_28")
campaign28_module = importlib.reload(campaign28_module)
dataloader_module = importlib.reload(dataloader_module)
CAMPAIGN28_SCROLL_DICT = campaign28_module.CAMPAIGN28_SCROLL_DICT
CAMPAIGN28_SCROLL_IDS = campaign28_module.CAMPAIGN28_SCROLL_IDS
TESTS = campaign28_module.TESTS
build_config = campaign28_module.build_config
DataManager = dataloader_module.DataManager
SAVE_PNG = True
OUT_DIR = ROOT / "output" / "campaign28_split_audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCROLL_NAMES = {
    20260115000000: "PHerc0139 w044",
    20260317000000: "PHerc0139 w035",
    20250223000000: "PHerc0139 w059",
    20251111010954: "PHerc0172 w068",
    20251112000002: "PHerc0172 w087",
    20240304141531: "PHerc1667 w013",
    20240304144031: "PHerc1667 w018",
    20231201215900: "PHerc1667 Cr1 Fr3",
    20250919125754: "PHerc0009B 487",
    20231210121321: "PHercParis4",
    20250628074500: "PHerc0500P2",
    20260226000000: "PHerc0814",
    20230301213755: "PHercParis2 Fr143",
    20231205222200: "PHerc51 Cr4 Fr8",
    20230301213423: "PHercParis1 Fr34",
    20250511003658: "PHerc0343P",
    20260221022814: "PHerc0841",
}
assert tuple(SCROLL_NAMES) == CAMPAIGN28_SCROLL_IDS

DOMAIN_BY_SCROLL = {
    scroll_id: domain_index
    for domain_index, scroll_ids in enumerate(CAMPAIGN28_SCROLL_DICT.values())
    for scroll_id in scroll_ids
}
COLORS = {
    "train_positive": np.array([255, 0, 0], dtype=np.uint8),
    "explicit_positive": np.array([255, 0, 220], dtype=np.uint8),
    "train_negative": np.array([255, 220, 0], dtype=np.uint8),
    "valid_positive": np.array([0, 90, 255], dtype=np.uint8),
    "valid_negative": np.array([0, 190, 70], dtype=np.uint8),
    "explicit_negative": np.array([0, 255, 255], dtype=np.uint8),
}


def campaign28_audit_config():
    config = build_config(TESTS[0])
    config.data.preload_volumes = False
    config.data.selective_chunk_preload = False
    config.data.mask_memmap = False
    config.data.mask_bitpack = False
    config.data.character_balanced_sampling = False
    config.tra.character_macro_metrics = False
    config.dl.data_aug = False
    return config


print(
    f"Campaign 28 audit ready: {len(CAMPAIGN28_SCROLL_IDS)} fragments, "
    f"{len(CAMPAIGN28_SCROLL_DICT)} physical domains"
)

In [ ]:
OVERLAY_ALPHA = 0.5


def _blend(rgb, selected, color):
    if not np.any(selected):
        return
    rgb[selected] = np.clip(
        (1.0 - OVERLAY_ALPHA) * rgb[selected].astype(np.float32)
        + OVERLAY_ALPHA * np.asarray(color, dtype=np.float32),
        0,
        255,
    ).astype(np.uint8)


def _empty_maps(shape):
    return {name: np.zeros(shape, dtype=bool) for name in COLORS}


def _collect_targets(dataset, maps, split):
    n = dataset._mt_grid
    sub = dataset._mt_sub
    explicit_positive_mask = dataset.explicit_positive_mask
    stats = {
        "windows": len(dataset.block_coords),
        "positive_occurrences": 0,
        "negative_occurrences": 0,
        "explicit_positive_occurrences": 0,
        "explicit_negative_occurrences": 0,
    }
    for _, y_off, x_off in dataset.block_coords:
        labels = dataset._fetch_label_mt(y_off, x_off).numpy()
        valid = dataset._fetch_mask_mt(y_off, x_off).numpy() > 0
        y0, _, x0, _ = dataset._mt_center_bounds(y_off, x_off)
        for index in np.flatnonzero(valid):
            iy, ix = divmod(int(index), n)
            ys, ye = y0 + iy * sub, y0 + (iy + 1) * sub
            xs, xe = x0 + ix * sub, x0 + (ix + 1) * sub
            cy = (ys + sub // 2) // sub
            cx = (xs + sub // 2) // sub
            if not (0 <= cy < maps["train_positive"].shape[0] and 0 <= cx < maps["train_positive"].shape[1]):
                continue
            if labels[index] < 0:
                maps["explicit_negative"][cy, cx] = True
                stats["explicit_negative_occurrences"] += 1
            elif labels[index] > 0:
                explicit = explicit_positive_mask is not None and np.any(
                    explicit_positive_mask[ys:ye, xs:xe] > 0
                )
                key = "explicit_positive" if explicit else f"{split}_positive"
                maps[key][cy, cx] = True
                stats["explicit_positive_occurrences" if explicit else "positive_occurrences"] += 1
            else:
                maps[f"{split}_negative"][cy, cx] = True
                stats["negative_occurrences"] += 1
    return stats


def render_scroll(scroll_id, name):
    config = campaign28_audit_config()
    scroll_index = list(CAMPAIGN28_SCROLL_IDS).index(int(scroll_id))
    manager = DataManager(
        config,
        scroll_id=int(scroll_id),
        domain_id=DOMAIN_BY_SCROLL[int(scroll_id)],
        character_namespace=scroll_index,
    )
    train_set, valid_set = manager.get_datasets()

    height, width = manager.labels.shape
    sub = int(config.model.multitile_subtile)
    shape = (int(np.ceil(height / sub)) + 1, int(np.ceil(width / sub)) + 1)
    maps = _empty_maps(shape)
    train_stats = _collect_targets(train_set, maps, "train")
    valid_stats = _collect_targets(valid_set, maps, "valid")

    train_any = maps["train_positive"] | maps["explicit_positive"] | maps["train_negative"] | maps["explicit_negative"]
    valid_any = maps["valid_positive"] | maps["valid_negative"]
    assert not np.any(train_any & valid_any), f"train/validation overlap for {scroll_id}"

    reference_path = ROOT / "inklabels" / "2_4um" / f"{scroll_id}.png"
    reference = cv2.imread(str(reference_path), cv2.IMREAD_GRAYSCALE)
    if reference is None:
        reference_path = ROOT / "inklabels" / f"{scroll_id}.png"
        reference = cv2.imread(str(reference_path), cv2.IMREAD_GRAYSCALE)
    background = cv2.resize(reference, (shape[1], shape[0]), interpolation=cv2.INTER_AREA)
    rgb = np.repeat(background[..., None], 3, axis=2).astype(np.uint8)
    for key in ("train_negative", "valid_negative", "train_positive", "valid_positive", "explicit_positive", "explicit_negative"):
        _blend(rgb, maps[key], COLORS[key])

    counts = {key: int(value.sum()) for key, value in maps.items()}
    figure, axis = plt.subplots(figsize=(18, max(6, 18 * height / width)))
    axis.imshow(rgb, interpolation="nearest")
    axis.set_title(
        f"{name} — {scroll_id}\n"
        "Campaign 28: ./inklabels, closed c2/g2/s4, pos_only=True; "
        "background=inklabels/2_4um"
    )
    axis.axis("off")
    axis.legend(handles=[
        mpatches.Patch(color=COLORS["train_positive"] / 255, alpha=OVERLAY_ALPHA, label=f"train positive ({counts['train_positive']:,})"),
        mpatches.Patch(color=COLORS["explicit_positive"] / 255, alpha=OVERLAY_ALPHA, label=f"explicit positive ({counts['explicit_positive']:,})"),
        mpatches.Patch(color=COLORS["train_negative"] / 255, alpha=OVERLAY_ALPHA, label=f"train ring negative ({counts['train_negative']:,})"),
        mpatches.Patch(color=COLORS["valid_positive"] / 255, alpha=OVERLAY_ALPHA, label=f"validation positive ({counts['valid_positive']:,})"),
        mpatches.Patch(color=COLORS["valid_negative"] / 255, alpha=OVERLAY_ALPHA, label=f"validation ring negative ({counts['valid_negative']:,})"),
        mpatches.Patch(color=COLORS["explicit_negative"] / 255, alpha=OVERLAY_ALPHA, label=f"explicit negative ({counts['explicit_negative']:,})"),
        mpatches.Patch(color=(0.5, 0.5, 0.5), label="researcher 2.4um reference"),
    ], loc="upper right", fontsize=9, framealpha=0.95)
    plt.tight_layout()

    output_path = OUT_DIR / f"{scroll_id}_campaign28_split.png"
    if SAVE_PNG:
        figure.savefig(output_path, dpi=180, bbox_inches="tight")
        print(f"saved {output_path}")
    print("unique targets", counts)
    print("train occurrences", train_stats)
    print("valid occurrences", valid_stats)
    plt.show()

    del train_set, valid_set, manager, reference, background, rgb, maps
    gc.collect()
    return counts


print("Campaign 28 renderer ready")

In [ ]:
render_scroll(20260115000000, SCROLL_NAMES[20260115000000])

In [ ]:
render_scroll(20260317000000, SCROLL_NAMES[20260317000000])

In [ ]:
render_scroll(20250223000000, SCROLL_NAMES[20250223000000])

In [ ]:
render_scroll(20251111010954, SCROLL_NAMES[20251111010954])

In [ ]:
render_scroll(20251112000002, SCROLL_NAMES[20251112000002])

In [ ]:
render_scroll(20240304141531, SCROLL_NAMES[20240304141531])

In [ ]:
render_scroll(20240304144031, SCROLL_NAMES[20240304144031])

In [ ]:
render_scroll(20231201215900, SCROLL_NAMES[20231201215900])

In [ ]:
render_scroll(20250919125754, SCROLL_NAMES[20250919125754])

In [ ]:
render_scroll(20231210121321, SCROLL_NAMES[20231210121321])

In [ ]:
render_scroll(20250628074500, SCROLL_NAMES[20250628074500])

In [ ]:
render_scroll(20260226000000, SCROLL_NAMES[20260226000000])

In [ ]:
render_scroll(20230301213755, SCROLL_NAMES[20230301213755])

In [ ]:
render_scroll(20231205222200, SCROLL_NAMES[20231205222200])

In [ ]:
render_scroll(20230301213423, SCROLL_NAMES[20230301213423])

In [ ]:
render_scroll(20250511003658, SCROLL_NAMES[20250511003658])

In [ ]:
render_scroll(20260221022814, SCROLL_NAMES[20260221022814])